In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from numpy.linalg import norm

RUN_DIR = Path("experiments/RN50_20250623_214602")   # adjust if needed

# embeddings
IMG_EMB = np.load(RUN_DIR / "img_embs.npy").astype("float32")
TXT_EMB = np.load(RUN_DIR / "txt_embs.npy").astype("float32")
IDS     = json.loads((RUN_DIR / "ids.json").read_text())

# metadata
PARQUET = Path(r"C:/Users/steph/OneDrive/Desktop/data/metadata.parquet")
META    = pd.read_parquet(PARQUET).set_index("id").loc[IDS]

# l2-normalise once
txt_norm = TXT_EMB / norm(TXT_EMB, axis=1, keepdims=True)

print("Loaded:", RUN_DIR.name, "| captions", txt_norm.shape[0])
META["domain"].value_counts()


Loaded: RN50_20250623_214602 | captions 2000


domain
coco      1368
flickr     401
sd         231
Name: count, dtype: int64

In [ ]:
import numpy as np, json

idx_coco    = META.index[META["domain"] == "coco"]
idx_flickr  = META.index[META["domain"] == "flickr"]

pos_coco    = META.index.get_indexer(idx_coco)      
pos_flickr  = META.index.get_indexer(idx_flickr)

coco_vecs   = txt_norm[pos_coco]
flickr_vecs = txt_norm[pos_flickr]

def recall_cross(source, target, k=1, chunk=1000):
    n = source.shape[0]
    hits = 0
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        sim = source[s:e] @ target.T               
        topk = np.argpartition(-sim, k-1, axis=1)[:, :k]
        hits += np.sum(topk == np.arange(e - s)[:, None])
    return hits / n * 100.0

metrics_dd = {
    "coco_flickr_R@1":  round(recall_cross(coco_vecs,   flickr_vecs, 1), 2),
    "flickr_coco_R@1":  round(recall_cross(flickr_vecs, coco_vecs,   1), 2),
}
out_file = RUN_DIR / "dataset2dataset_metrics.json"
json.dump(metrics_dd, open(out_file, "w"), indent=2)
print("Saved", out_file, metrics_dd)



Saved experiments\RN50_20250623_214602\dataset2dataset_metrics.json {'coco_flickr_R@1': 0.15, 'flickr_coco_R@1': 0.0}
